In [14]:
# ensure connection to cluster
spark

In [16]:
# Cell 1 — kill the existing session
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
spark.stop()
print("Stopped")

Stopped


In [17]:
# load packages
from pyspark.sql import SparkSession
import pandas as pd
from pyspark.sql import functions as F

# Cell 2 — start fresh in local mode
spark = SparkSession.builder \
    .appName("GTEx Full Load") \
    .master("local[4]") \
    .config("spark.driver.memory", "24g") \
    .config("spark.sql.parquet.mergeSchema", "false") \
    .config("spark.sql.parquet.filterPushdown", "true") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.driver.maxResultSize", "8g") \
    .getOrCreate()

print(spark.sparkContext.master)  # should print "local[4]"

26/05/29 01:49:15 INFO SparkEnv: Registering MapOutputTracker
26/05/29 01:49:15 INFO SparkEnv: Registering BlockManagerMaster
26/05/29 01:49:15 INFO SparkEnv: Registering BlockManagerMasterHeartbeat
26/05/29 01:49:15 INFO SparkEnv: Registering OutputCommitCoordinator


local[4]


----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 58140)
Traceback (most recent call last):
  File "/opt/conda/miniconda3/lib/python3.10/socketserver.py", line 316, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/opt/conda/miniconda3/lib/python3.10/socketserver.py", line 347, in process_request
    self.finish_request(request, client_address)
  File "/opt/conda/miniconda3/lib/python3.10/socketserver.py", line 360, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/opt/conda/miniconda3/lib/python3.10/socketserver.py", line 747, in __init__
    self.handle()
  File "/usr/lib/spark/python/pyspark/accumulators.py", line 281, in handle
    poll(accum_updates)
  File "/usr/lib/spark/python/pyspark/accumulators.py", line 253, in poll
    if func():
  File "/usr/lib/spark/python/pyspark/accumulators.py", line 257, in accum_updates
    num_updates = read_int(self.r

In [ ]:
# load data into memory (4GB takes ~30 sec)
df = spark.read.parquet("gs://gene_datasets/GTEx_tissue_expression.parquet")
print((df.count(), len(df.columns)))

(74628, 19618)


In [25]:
# Read sample attributes to link codes to tissues
# Step 2: Load tissue attributes
attrs = spark.createDataFrame(
    pd.read_csv(
        "https://storage.googleapis.com/adult-gtex/annotations/v11/metadata-files/GTEx_Analysis_v11_Annotations_SampleAttributesDS.txt",
        sep="\t",
        usecols=["SAMPID", "SMTSD"]
    )
)

In [27]:
from pyspark.sql import functions as F

# Step 3: Unpivot wide to long format
#         (Spark 3.4+ has stack() or unpivot(); Dataproc likely has 3.3 so use stack)
sample_cols = [c for c in df.columns if c.startswith("GTEX")]

stack_expr = "stack({}, {}) as (sample_id, tpm)".format(
    len(sample_cols),
    ", ".join([f"'{c}', `{c}`" for c in sample_cols])
)

df_long = df.select(
    F.expr("Name").alias("gene_id"),
    F.expr(stack_expr)
)

# Step 4: Join tissue labels
df_long = df_long.join(attrs, df_long.sample_id == attrs.SAMPID, "left") \
                 .drop("SAMPID")

# Step 5: Compute median TPM per gene per tissue
df_median = df_long.groupBy("gene_id", "SMTSD") \
                   .agg(F.percentile_approx("tpm", 0.5).alias("median_tpm"))

# Step 6: Pivot back to wide format (genes × tissues)
df_wide = df_median.groupBy("gene_id") \
                   .pivot("SMTSD") \
                   .agg(F.first("median_tpm"))

# Strip version suffix
df_wide = df_wide.withColumn("gene_id", F.split(F.col("gene_id"), r"\.")[0])

df_wide.cache()
df_wide.show(5)

26/05/29 02:05:14 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/05/29 02:07:15 WARN DAGScheduler: Broadcasting large task binary with size 3.6 MiB
26/05/29 02:09:01 ERROR Executor: Exception in task 31.0 in stage 4.0 (TID 70)2]
java.lang.OutOfMemoryError: Java heap space
	at org.apache.spark.sql.execution.vectorized.OnHeapColumnVector.reserveInternal(OnHeapColumnVector.java:578) ~[spark-sql_2.12-3.3.2.jar:3.3.2]
	at org.apache.spark.sql.execution.vectorized.OnHeapColumnVector.<init>(OnHeapColumnVector.java:79) ~[spark-sql_2.12-3.3.2.jar:3.3.2]
	at org.apache.spark.sql.execution.vectorized.OnHeapColumnVector.allocateColumns(OnHeapColumnVector.java:53) ~[spark-sql_2.12-3.3.2.jar:3.3.2]
	at org.apache.spark.sql.execution.vectorized.OnHeapColumnVector.allocateColumns(OnHeapColumnVector.java:42) ~[spark-sql_2.12-3.3.2.jar:3.3.2]
	at org.apache.spark.sql.execution.datasource

Py4JJavaError: An error occurred while calling o279.pivot.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 31 in stage 4.0 failed 1 times, most recent failure: Lost task 31.0 in stage 4.0 (TID 70) (mycluster-m.c.gene-expression-big-data.internal executor driver): java.lang.OutOfMemoryError: Java heap space
	at org.apache.spark.sql.execution.vectorized.OnHeapColumnVector.reserveInternal(OnHeapColumnVector.java:578)
	at org.apache.spark.sql.execution.vectorized.OnHeapColumnVector.<init>(OnHeapColumnVector.java:79)
	at org.apache.spark.sql.execution.vectorized.OnHeapColumnVector.allocateColumns(OnHeapColumnVector.java:53)
	at org.apache.spark.sql.execution.vectorized.OnHeapColumnVector.allocateColumns(OnHeapColumnVector.java:42)
	at org.apache.spark.sql.execution.datasources.parquet.VectorizedParquetRecordReader.initBatch(VectorizedParquetRecordReader.java:271)
	at org.apache.spark.sql.execution.datasources.parquet.VectorizedParquetRecordReader.initBatch(Vectoriz